In [22]:
!pip install pycountry

import pandas as pd
import numpy as np
import requests
import pycountry
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [23]:
data = pd.read_excel("195_final_cleaned_dataset.xlsx")
data

,Country,Year,PM25,LifeExp,GDP,HealthExp,log_PM25,log_GDP,log_HealthExp
0,Afghanistan,2020,46.087094,61.454,2561.981761,401.163333,3.830533,7.848536,5.994369
1,Afghanistan,2019,58.330872,62.941,2583.485332,383.164896,4.066131,7.856895,5.948465
2,Afghanistan,2018,67.227177,62.443,2432.276701,345.587947,4.208078,7.796583,5.845247
3,Afghanistan,2017,65.862347,62.406,2335.795862,294.796437,4.187567,7.756108,5.686285
4,Afghanistan,2016,72.765910,62.646,2213.181441,261.566867,4.287248,7.702186,5.566690
...,...,...,...,...,...,...,...,...,...
3411,Zimbabwe,2014,23.429253,58.106,3903.427977,203.300574,3.153985,8.269610,5.314686
3412,Zimbabwe,2013,22.598401,56.842,3783.946337,173.586892,3.117879,8.238523,5.156678
3413,Zimbabwe,2012,23.539528,55.386,3472.485720,156.086614,3.158681,8.152626,5.050411
3414,Zimbabwe,2011,23.616283,53.911,3047.317089,161.209011,3.161936,8.022017,5.082702


In [24]:
# baseline model
X1 = sm.add_constant(data[["log_PM25"]])
Y = data["LifeExp"]

model1 = sm.OLS(Y, X1).fit(cov_type="HC1")
print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:                LifeExp   R-squared:                       0.183
Model:                            OLS   Adj. R-squared:                  0.183
Method:                 Least Squares   F-statistic:                     666.9
Date:                Wed, 01 Apr 2026   Prob (F-statistic):          1.80e-134
Time:                        00:53:26   Log-Likelihood:                -11997.
No. Observations:                3416   AIC:                         2.400e+04
Df Residuals:                    3414   BIC:                         2.401e+04
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         89.5062      0.736    121.571      0.0

In [25]:
# model 2: economic controls
X2 = sm.add_constant(data[["log_PM25", "log_GDP", "log_HealthExp"]])

model2 = sm.OLS(Y, X2).fit(cov_type="HC1")
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                LifeExp   R-squared:                       0.704
Model:                            OLS   Adj. R-squared:                  0.704
Method:                 Least Squares   F-statistic:                     3355.
Date:                Wed, 01 Apr 2026   Prob (F-statistic):               0.00
Time:                        00:53:28   Log-Likelihood:                -10263.
No. Observations:                3416   AIC:                         2.053e+04
Df Residuals:                    3412   BIC:                         2.056e+04
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            26.7415      1.118     23.923

In [36]:
# train-test data split (did training and testing on second model -- the controlled model)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X2, Y, test_size=0.2, random_state=42)

# fit model to train
model_train = sm.OLS(y_train, X_train).fit()

In [38]:
# model testing
y_pred = model_train.predict(X_test)

In [39]:
# test evaluation metrics
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Test R2:", r2)
print("Test RMSE:", rmse)

Test R2: 0.6866630628703283
Test RMSE: 4.993519593172892


In [40]:
# train error
r2_train = model_train.rsquared
y_train_pred = model_train.predict(X_train)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))

print("Train R2:", r2_train)
print("Train RMSE:", train_rmse)

Train R2: 0.707870370959403
Train RMSE: 4.853661073075797
